In [0]:
# Source path for CSV files
source_path = "/Volumes/retail_q/volume_schema/blob_storage/transactions/"

# Target table
target_table = "retail_q.blob_bronze.transactions"

# Checkpoint location for Auto Loader
checkpoint_path = "/Volumes/retail_q/volume_schema/blob_storage/_checkpoints/transactions_bronze"

# Read CSV files using Auto Loader
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path))

# Write to bronze table and wait for completion
query = (df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table))

# Wait for the stream to process all available data and stop
query.awaitTermination()

In [0]:
%sql
select count(*) from retail_q.blob_bronze.transactions;